# GitHub Triager RL Training
## Training Llama-3.2-3B to triage GitHub issues using GRPO + TRL + Unsloth

**Environment:** https://huggingface.co/spaces/Kavya011/github-triager-rl  
**GitHub:** https://github.com/KavyaTejani/Github-Triager

In [ ]:
!pip install unsloth trl transformers datasets peft accelerate matplotlib
!pip install git+https://huggingface.co/spaces/Kavya011/github-triager-rl

In [ ]:
ENVIRONMENT_URL = "https://kavya011-github-triager-rl.hf.space"
MODEL_NAME      = "unsloth/Llama-3.2-3B-Instruct"
MAX_STEPS       = 200      # increase to 500+ for real training
BATCH_SIZE      = 4
NUM_GENERATIONS = 4        # GRPO rollouts per prompt

In [ ]:
import asyncio
try:
    from github_triager.client import GitHubTriagerClient
except ImportError:
    # Fallback if package structure differs
    try:
        from client import GitHubTriagerClient
    except ImportError:
        print("Client import failed. Ensure environment package is installed.")

async def test_connection():
    async with GitHubTriagerClient(base_url=ENVIRONMENT_URL) as env:
        result = await env.reset(task_id="label_classification")
        print("Connected! Observation keys:", list(result.keys()))
        return result

try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

obs = asyncio.run(test_connection())
print(obs)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model loaded successfully.")

In [ ]:
async def rollout_single(task_id: str = "label_classification"):
    async with GitHubTriagerClient(base_url=ENVIRONMENT_URL) as env:
        obs = await env.reset(task_id=task_id)
        prompt = f"""You are a GitHub issue triager.
Issue Title: {obs['issue']['title']}
Issue Body: {obs['issue']['body']}
Task: Classify this issue. Respond with valid JSON only.
Format: {{"label": "<bug|feature|documentation|question|enhancement>"}}"""

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.7)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = response[len(prompt):]   # strip prompt from output

        result = await env.step({"response": response})
        return prompt, response, float(result.get("reward", 0.0))

# Sanity check
try:
    prompt, response, reward = asyncio.run(rollout_single())
    print(f"Reward: {reward:.3f}")
    print(f"Model response: {response}")
except Exception as e:
    print(f"Rollout failed (expected if not on GPU): {e}")

In [ ]:
async def get_env_reward(completion: str) -> float:
    async with GitHubTriagerClient(base_url=ENVIRONMENT_URL) as env:
        await env.reset(task_id="label_classification")
        result = await env.step({"response": completion})
        return float(result.get("reward", 0.0))

def compute_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        reward = asyncio.run(get_env_reward(text))
        rewards.append(reward)
    return rewards

In [ ]:
from datasets import Dataset

async def build_dataset(n_samples: int = 100):
    samples = []
    async with GitHubTriagerClient(base_url=ENVIRONMENT_URL) as env:
        for i in range(n_samples):
            obs = await env.reset(task_id="label_classification")
            prompt = f"""You are a GitHub issue triager.
Issue Title: {obs['issue']['title']}
Issue Body: {obs['issue']['body']}
Classify this issue. Respond with JSON only.
Format: {{"label": "<bug|feature|documentation|question|enhancement>"}}"""
            samples.append({"prompt": prompt})
    return Dataset.from_list(samples)

dataset = asyncio.run(build_dataset(100))
print(f"Dataset ready: {len(dataset)} samples")

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="./outputs/github-triager-grpo",
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=NUM_GENERATIONS,
    max_steps=MAX_STEPS,
    learning_rate=5e-6,
    logging_steps=10,
    save_steps=50,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[compute_reward],
    tokenizer=tokenizer,
)

trainer.train()
print("Training complete.")

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("results", exist_ok=True)

log_history = trainer.state.log_history
steps        = [x["step"]   for x in log_history if "loss"   in x]
losses       = [x["loss"]   for x in log_history if "loss"   in x]
r_steps      = [x["step"]   for x in log_history if "reward" in x]
rewards      = [x["reward"] for x in log_history if "reward" in x]

# Loss curve
plt.figure(figsize=(10, 4))
plt.plot(steps, losses, color="royalblue", linewidth=2)
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("GitHub Triager — Training Loss (GRPO)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/loss_curve.png", dpi=150)
plt.show()
print("Saved: results/loss_curve.png")

# Reward curve
plt.figure(figsize=(10, 4))
plt.plot(r_steps, rewards, color="seagreen", linewidth=2)
plt.xlabel("Training Step")
plt.ylabel("Average Reward")
plt.title("GitHub Triager — Reward During Training (GRPO)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/reward_curve.png", dpi=150)
plt.show()
print("Saved: results/reward_curve.png")

In [ ]:
async def evaluate_model(n_episodes=20):
    total = 0.0
    async with GitHubTriagerClient(base_url=ENVIRONMENT_URL) as env:
        for _ in range(n_episodes):
            obs = await env.reset(task_id="label_classification")
            prompt = f"Classify this GitHub issue.\nTitle: {obs['issue']['title']}\nBody: {obs['issue']['body']}\nJSON:"
            inputs  = tokenizer(prompt, return_tensors="pt").to("cuda")
            outputs = model.generate(**inputs, max_new_tokens=64)
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):]
            result = await env.step({"response": response})
            total += float(result.get("reward", 0.0))
    return total / n_episodes

try:
    trained_avg = asyncio.run(evaluate_model(20))
    baseline    = 0.20   # replace with actual pre-training score if measured
    print(f"Baseline avg reward : {baseline:.3f}")
    print(f"Trained  avg reward : {trained_avg:.3f}")

    plt.figure(figsize=(6, 5))
    plt.bar(["Baseline\n(Untrained)", "Trained\n(GRPO)"],
            [baseline, trained_avg],
            color=["#ff6b6b", "#51cf66"], edgecolor="black", width=0.5)
    plt.ylabel("Average Reward (label_classification)")
    plt.title("Before vs After GRPO Training")
    plt.ylim(0, 1.05)
    for i, v in enumerate([baseline, trained_avg]):
        plt.text(i, v + 0.02, f"{v:.2f}", ha="center", fontweight="bold", fontsize=13)
    plt.tight_layout()
    plt.savefig("results/before_after_comparison.png", dpi=150)
    plt.show()
    print("Saved: results/before_after_comparison.png")
except Exception as e:
    print(f"Evaluation failed: {e}")

In [ ]:
model.save_pretrained("outputs/github-triager-adapter")
tokenizer.save_pretrained("outputs/github-triager-adapter")
print("Adapter saved. Do NOT upcast 4-bit model and merge — use adapter directly.")